Question 7 : Return the third transaction (by transaction_date) of each user. Users with fewer than three transactions don't appear. The output columns are user_id, spend and transaction_date, ordered by user_id

In [1]:
import pandas as pd

transactions = pd.DataFrame({
    "user_id": [111, 131, 111, 121, 131, 111, 131, 121, 111, 131],
    "spend": [100.50, 75.00, 55.00, 20.00, 10.00, 300.00, 60.00, 40.00, 25.00, 90.00],
    "transaction_date": pd.to_datetime([
        "2022-01-08 09:00", "2022-03-05 08:00", "2022-01-10 11:00", "2022-02-01 10:00",
        "2022-03-01 08:00", "2022-01-15 13:00", "2022-03-03 08:00", "2022-02-03 10:00",
        "2022-01-12 12:00", "2022-03-09 08:00"]),
})

In [14]:
transactions = transactions.sort_values(['user_id', 'transaction_date'])

transactions['rnk'] = transactions.groupby('user_id')['transaction_date'].rank(method='first')

# transactions['max_rnk'] = transactions.groupby('user_id')['rnk'].transform('max')

txn_ge_3 = transactions.loc[(transactions['rnk'] == 3)]

txn_3rd = txn_ge_3[['user_id', 'spend', 'transaction_date']].reset_index(drop=True)

txn_3rd

,user_id,spend,transaction_date
0,111,25.0,2022-01-12 12:00:00
1,131,75.0,2022-03-05 08:00:00


Question 8: Return employee_id and name of every employee who earns more than their own manager, ordered by employee_id. Employees with no manager don't qualify.

In [15]:
import pandas as pd

employee = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5, 6, 7],
    "name": ["Asha", "Ben", "Chitra", "Dev", "Esha", "Farid", "Gita"],
    "salary": [5200, 3100, 9000, 7000, 2800, 12000, 6500],
    "manager_id": [4, 5, 4, 6, 6, None, 5],
})

In [23]:
emp_man = employee.merge(employee, how = 'left', left_on = 'manager_id', right_on = 'employee_id')

emp_man_trimmed = emp_man[['employee_id_x', 'name_x', 'name_y', 'salary_x', 'salary_y']]

emp_man_trimmed = emp_man_trimmed.rename(columns={
    'employee_id_x': 'employee_id',
    'name_x': 'employee_name',
    'name_y': 'manager_name',
    'salary_x': 'employee_salary',
    'salary_y': 'manager_salary',
})

emp_man_trimmed = emp_man_trimmed[~ emp_man_trimmed['manager_name'].isna()]

emp_ge_man = emp_man_trimmed.loc[emp_man_trimmed['employee_salary'] > emp_man_trimmed['manager_salary']]

emp_ge_man[['employee_id', 'employee_name']].sort_values('employee_id').reset_index(drop=True)

,employee_id,employee_name
0,2,Ben
1,3,Chitra
2,7,Gita


Question 9: For each user, find the longest run of consecutive calendar days with at least one login. A user who logged in several times on one day counts that day once. Return user_id and longest_streak (in days), ordered by user_id

In [24]:
import pandas as pd

logins = pd.DataFrame({
    "user_id": [1, 3, 1, 2, 3, 1, 3, 1, 2, 3, 1, 3, 2, 1, 3],
    "login_date": pd.to_datetime([
        "2022-03-05", "2022-03-04", "2022-03-01", "2022-03-12", "2022-03-01",
        "2022-03-03", "2022-03-06", "2022-03-02", "2022-03-10", "2022-03-02",
        "2022-03-06", "2022-03-05", "2022-03-14", "2022-03-06", "2022-03-07"]),
})

In [38]:
logins_dedup = logins.drop_duplicates(['user_id', 'login_date']).sort_values(['user_id', 'login_date']).reset_index(drop = True)

logins_dedup['lag_date'] = logins_dedup.groupby('user_id')['login_date'].shift(1)

logins_dedup['datediff'] = (logins_dedup['login_date'] - logins_dedup['lag_date']).dt.days

logins_dedup['new_streak'] = (logins_dedup['datediff'] != 1).astype(int)

logins_dedup['streak_id'] = logins_dedup.groupby('user_id')['new_streak'].cumsum()

island_size = logins_dedup.groupby(['user_id', 'streak_id']).size()

max_island_size = island_size.groupby('user_id').max().reset_index(name='longest_streak')

max_island_size

,user_id,longest_streak
0,1,3
1,2,1
2,3,4


Question 10: For each app, compute the click-through rate for 2022 only: 100.0 * clicks / impressions, rounded to 2 decimals. Return app_id and ctr, ordered by app_id.

In [39]:
import pandas as pd

events = pd.DataFrame({
    "app_id": [123, 123, 123, 123, 123, 123, 234, 234, 234, 234, 234, 234, 345, 345],
    "event_type": ["impression", "impression", "click", "impression", "impression", "click",
                   "impression", "impression", "click", "impression", "impression", "click",
                   "impression", "impression"],
    "timestamp": pd.to_datetime([
        "2022-07-18 11:36", "2022-07-18 11:37", "2022-07-18 11:37", "2022-08-01 09:00",
        "2022-08-02 10:00", "2021-12-31 23:59", "2022-06-01 08:00", "2022-06-02 08:00",
        "2022-06-02 08:05", "2022-06-03 08:00", "2021-11-11 08:00", "2021-11-11 08:01",
        "2022-02-02 12:00", "2022-02-03 12:00"]),
})

In [45]:
events_2022 = events.loc[events['timestamp'].dt.year == 2022].copy()

events_2022['impression_flag'] = (events_2022['event_type'] == 'impression').astype(int)
events_2022['click_flag'] = (events_2022['event_type'] == 'click').astype(int)

events_2022_agg = events_2022.groupby('app_id').agg(impressions = ('impression_flag', 'sum'), clicks = ('click_flag', 'sum')).reset_index()

events_2022_agg['ctr'] = (events_2022_agg['clicks']*100.0/events_2022_agg['impressions']).round(2)

events_2022_agg[['app_id', 'ctr']]

,app_id,ctr
0,123,25.00
1,234,33.33
2,345,0.00


these can be used too

events_2022['impression_flag'] = np.where(events_2022['event_type'] == 'impression', 1, 0)  
events_2022['click_flag'] = np.where(events_2022['event_type'] == 'click', 1, 0)

some more syntax ideas for case when type implementation:

events['click_flag'] = np.where(events['event_type'] == 'click', 1, 0)  
events['period'] = np.select(
    [events['timestamp'].dt.year == 2022, events['timestamp'].dt.year == 2021],
    ['current', 'previous'],
    default='older'
)

Question 11: Return the median number of searches per user across all users, rounded to 1 decimal place, as a one-row DataFrame with a single column named median.

In [1]:
import pandas as pd

search_frequency = pd.DataFrame({
    "searches":  [3, 1, 4, 2],
    "num_users": [3, 2, 1, 2],
})

In [6]:
import numpy as np

np.median(np.repeat(search_frequency['searches'], search_frequency['num_users']))

np.float64(2.5)

Question 12: For each user and date, compute the rolling average of tweet_count over that day and the two days before it (a window of 3 days). For a user's first two days, average whatever days are available. Round to 2 decimals. Return user_id, tweet_date and rolling_avg_3d, ordered by user_id, then tweet_date. Every user has tweets on consecutive days, with no gaps

In [7]:
import pandas as pd

tweets = pd.DataFrame({
    "user_id": [111, 112, 111, 111, 112, 111, 112, 111],
    "tweet_date": pd.to_datetime([
        "2022-06-03", "2022-06-02", "2022-06-01", "2022-06-05",
        "2022-06-01", "2022-06-02", "2022-06-03", "2022-06-04"]),
    "tweet_count": [3, 6, 2, 5, 10, 1, 2, 4],
})

In [15]:
tweet_sorted = tweets.sort_values(['user_id', 'tweet_date'])

tweet_sorted['mvg_avg'] = tweet_sorted.groupby('user_id')['tweet_count'].rolling(3, min_periods = 1).mean().reset_index(level = 'user_id', drop = True)

tweet_sorted['mvg_avg'] = tweet_sorted['mvg_avg'].round(2)

tweet_sorted[['user_id', 'tweet_date', 'mvg_avg']].reset_index(drop=True)

,user_id,tweet_date,mvg_avg
0,111,2022-06-01,2.00
1,111,2022-06-02,1.50
2,111,2022-06-03,2.00
3,111,2022-06-04,2.67
4,111,2022-06-05,4.00
5,112,2022-06-01,10.00
6,112,2022-06-02,8.00
7,112,2022-06-03,6.00


min_periods

A rolling window of size 3 is "full" only when it has 3 values. min_periods is the smallest number of values the window needs before pandas returns a result instead of NaN. The default equals the window size.

On the series [2, 1, 3, 4, 5] with a window of 3:

Setting	                    Result	                    Meaning  
rolling(3)	                NaN, NaN, 2.0, 2.67, 4.0	Needs 3 values, so the first two are NaN  
rolling(3, min_periods=2)	NaN, 1.5, 2.0, 2.67, 4.0	Needs 2, so the second value gets an answer  
rolling(3, min_periods=1)	2.0, 1.5, 2.0, 2.67, 4.0	Needs 1, so every row gets an average  